# **Diabet Detection**

In [ ]:
# Installing neccessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import kagglehub
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
            accuracy_score,
            precision_score,
            recall_score,
            f1_score,
            confusion_matrix,
            classification_report,
            roc_curve,
            auc,
        )

In [ ]:
# relative path to the file
relative_dir = './diabetes.csv'

In [ ]:
df = pd.read_csv(relative_dir)
df.head()

In [ ]:
df.tail()

In [ ]:
print("Shape:", df.shape)
print(df.columns.tolist())
df.info()
df.describe().T

In [ ]:
# Check if there is missing value in the dataframe
df.isnull().sum()

In [ ]:
# Insulin. Target column in this CSV is "Outcome" (0 = No Diabetes, 1 = Diabetes)

features = ["Glucose", "BloodPressure", "BMI", "Age", "Insulin"]
target = "Outcome"

X = df[features].copy()  # copy it, bc if X changed, df must not be change too
y = df[target].copy()

X.head()

# Visualizing and Understanding the Data

In [ ]:
# Histograms
X.hist(bins=25, figsize=(12, 8))
plt.suptitle("Distributions of selected features")
plt.show()

In [ ]:
# Pairplot colored by Outcome (small sample if large)
sample_df = pd.concat([X, y], axis=1)
sns.pairplot(sample_df, hue=target, vars=features, diag_kind="hist", corner=True, plot_kws={"alpha": 0.6})
plt.suptitle("Pairplot of features (colored by Outcome)", y=1.02)
plt.show()


In [ ]:
# Correlation heatmap->how the columns relat each other
corr = pd.concat([X, y], axis=1).corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix")
plt.show()

In [ ]:
corr_with_target = corr[target].sort_values(ascending=False)
print("Order of Effect on the Target feature: Diabet")
print("Correlation with target:\n", corr_with_target)

In [ ]:
# Count zeros per feature
zero_counts = (X == 0).sum().sort_values(ascending=False)
print("Zero counts per feature:\n", zero_counts)

# Features where zero is not a valid measurement
invalid_zero_features = ["Glucose", "BloodPressure", "BMI", "Insulin"]


X_clean = X.copy()
for col in invalid_zero_features:
    nonzero_median = X_clean.loc[X_clean[col] != 0, col].median()
    X_clean.loc[X_clean[col] == 0, col] = nonzero_median
    print(f"Replaced zeros in {col} with median = {nonzero_median}")


X_clean.describe().T

In [ ]:
# 7) Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_scaled, y_train)

In [ ]:
# Evaluate on test set
y_pred = gb.predict(X_test_scaled)
y_proba = gb.predict_proba(X_test_scaled)[:, 1]  # probability of class 1


# Metrics Evaluation of the model

In [ ]:

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

In [ ]:

print("Test set metrics:")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\nClassification report:\n")
print(classification_report(y_test, y_pred, digits=4))

## Testing based on the new single dataset

In [ ]:
# Testing based on the new single dataset

def new_prediction(values_array):
    values_array = values_array.reshape(1, -1)
    values_array_scaled = scaler.transform(values_array)
    prediction = gb.predict(values_array_scaled)
    if prediction[0] == 0:
        print("No Diabets")
        print("but you can check hearbay hospital if feeling some simptoms!")
    else:
        print("Diabetic")
        print("A person with this data has Diabetes, we recommend you to check near by hospital")

new_prediction(np.array([150, 72, 33.6, 45, 120]))
